In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import dotenv
import os

config = dotenv.dotenv_values("../.env")
replace_keys = ["JAVA_HOME", "FUSEKI_HOME"]
append_keys = ["PATH"]
for key, value in config.items():
    # append to os.environ
    if key in append_keys:
        os.environ[key] = f"{os.environ.get(key, '')}:{value}"
    elif key in replace_keys:
        os.environ[key] = value


In [3]:
import sys
sys.path.append("../../benchmarks")

In [4]:
from utils.datasets.utils import parse_nt_to_generator, save_from_generator
from utils.datasets.dbpedia import DBPedia
from pathlib import Path

In [5]:
dataset=DBPedia(
    base_dir=Path("../data/dbpedia")
)

In [6]:
dataset.full_ttl_file

PosixPath('../data/dbpedia/dbpedia_complete.nt')

In [7]:
from utils.dbs.base_db import  BaseDB
from utils.dbs.qlever_native import QleverDBNative

db = QleverDBNative(
    dataset=dataset,
    id="dbpedia-encoded-tidx",
    name="QleverDB-DBPedia",
    base_dir=Path("../data/dbpedia"),
    use_encoded_ttl=False,
)
db.id = "dbpedia-encoded-tidx"
db.db_dir = Path("../data/dbpedia/index-encoded")

2026-08-25 11:12:54,331 - INFO - Loading faiss with AVX512 support.
2026-08-25 11:12:54,332 - INFO - Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-08-25 11:12:54,332 - INFO - Loading faiss with AVX2 support.
2026-08-25 11:12:54,333 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-08-25 11:12:54,333 - INFO - Loading faiss.


2026-08-25 11:12:54,364 - INFO - Successfully loaded faiss.
2026-08-25 11:12:54,375 - WARNING - Killing any existing process using port 26043 before starting the server
2026-08-25 11:12:54,703 - ERROR - Command failed with return code 1
2026-08-25 11:12:54,704 - INFO - Initialized QLeverDBNative with id=dbpedia-encoded-tidx, port_id=26043, dataset=DBPedia, name=QleverDB-DBPedia, use_encoded_ttl=False, endpoint=http://localhost:26043/dbpedia-encoded-tidx-with-tidx/sparql


In [8]:
db.server_log_file, db.db_dir

(PosixPath('../data/dbpedia/db/dbpedia-encoded-tidx-with-tidx/dbpedia-encoded-tidx-with-tidx_run.log'),
 PosixPath('../data/dbpedia/index-encoded'))

In [9]:
db.setup()

2026-08-25 11:12:54,797 - INFO - Setting up QLeverDBNative db_dir=../data/dbpedia/index-encoded, base_dir=../data/dbpedia
2026-08-25 11:12:54,798 - INFO - Logging QLever setup to ../data/dbpedia/db/dbpedia-encoded-tidx-with-tidx/dbpedia-encoded-tidx-with-tidx_run.log
2026-08-25 11:12:54,799 - INFO - Loading dataset into QLever server from ../data/dbpedia/dbpedia_complete.nt
2026-08-25 11:12:54,800 - WARNING - DB directory ../data/dbpedia/index-encoded already exists!
2026-08-25 11:12:54,800 - INFO - Stopping server!


2026-08-25 11:12:54,857 - ERROR - Command failed with return code 1
2026-08-25 11:12:54,858 - INFO - Starting QLever server on port 26043
2026-08-25 11:12:54,859 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 26043 -k 0 -m 32G --tensor-search-max-num-threads 4
2026-08-25 11:12:54,871 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/dbpedia-encoded-tidx-with-tidx/sparql)
2026-08-25 11:12:55,873 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/dbpedia-encoded-tidx-with-tidx/sparql)
2026-08-25 11:12:56,875 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/dbpedia-encoded-tidx-with-tidx/sparql)
2026-08-25 11:12:57,890 - INFO - Server is up and responding to queries


In [10]:
db.query("""SELECT (COUNT(?s) as ?count)
WHERE {
    ?s dbo:thumbnail_embedding ?thumbnail .
}""")

,count
0,2369261


In [11]:
dataset.left, dataset.right

('NaturalPlace', 'Village')

In [12]:
class_ranks = db.query("""SELECT ?cls  (COUNT(?cls) as ?count)
WHERE {
    ?s a ?cls;
     dbo:thumbnail_embedding ?thumbnail .
    FILTER(STRSTARTS(STR(?cls), "http://dbpedia.org/ontology/"))
} GROUP BY ?cls ORDER BY DESC(?count)""")
class_ranks.to_csv("../data/dbpedia/thumbnail_embedding_count_per_cls.csv", index=False)
class_ranks

,cls,count
0,dbo:Place,435458
1,dbo:Location,435458
2,dbo:Species,350010
3,dbo:Eukaryote,349864
4,dbo:Animal,347640
...,...,...
468,dbo:AnimangaCharacter,1
469,dbo:Sound,1
470,dbo:Conifer,1
471,dbo:SpeedwayLeague,1


In [13]:
lr_counts = db.query(f"""SELECT ?cls  (COUNT(?cls) as ?count)
WHERE {{
    ?s a ?cls;
     dbo:thumbnail_embedding ?thumbnail .
    FILTER(STRSTARTS(STR(?cls), "http://dbpedia.org/ontology/") && ?cls IN(dbo:{dataset.left}, dbo:{dataset.right}))

}} GROUP BY ?cls ORDER BY DESC(?count)""")
lr_counts

,cls,count
0,dbo:NaturalPlace,46351
1,dbo:Village,31149


In [14]:
cross_counts = db.query(f"""SELECT (COUNT(*) as ?count)
WHERE {{
    ?l a dbo:{dataset.left};
     dbo:thumbnail_embedding ?tl .
    ?r a dbo:{dataset.right};
     dbo:thumbnail_embedding ?tr .

}}""")
cross_counts

,count
0,1443787299


In [15]:
inter_counts = db.query(f"""SELECT (COUNT(*) as ?count)
WHERE {{
    ?lr a dbo:{dataset.left};
     dbo:thumbnail_embedding ?tl .
    ?lr a dbo:{dataset.right};
     dbo:thumbnail_embedding ?tr .
     }}""")
inter_counts

,count
0,0
